# 02. 线程、进程与 GIL

本章目标：

- 使用 `ThreadPoolExecutor` 并发执行阻塞式 I/O；
- 识别共享可变状态导致的竞态条件，并选择合适的同步工具；
- 理解默认 CPython 中 GIL 的影响；
- 使用 `ProcessPoolExecutor` 处理纯 Python CPU 密集型任务。

## 1. Python 线程是什么

`threading.Thread` 对应操作系统线程，与 Java `Thread` 类似。多个线程共享同一进程的内存，因此传递对象方便，但共享可变状态也需要同步。

线程适合包装已有的阻塞式 I/O API，例如同步数据库驱动或文件操作。等待 I/O 时，操作系统可以调度其他线程。

In [17]:
import threading


def show_thread() -> None:
    current = threading.current_thread()
    print(f"线程名称={current.name!r}, id={current.ident}")


show_thread()

线程名称='MainThread', id=8737038912


## 2. 用线程池重叠 I/O 等待

Java 通常通过 `ExecutorService` 管理线程池；Python 对应的高层 API 是 `ThreadPoolExecutor`。优先使用线程池，而不是为每个任务手动创建和销毁线程。

In [18]:
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter, sleep


def simulated_io(task_id: int, delay: float = 0.2) -> str:
    sleep(delay)
    return f"任务 {task_id} 完成"


started = perf_counter()
sequential = [simulated_io(index) for index in range(3)]
sequential_elapsed = perf_counter() - started

started = perf_counter()
with ThreadPoolExecutor(max_workers=3) as executor:
    concurrent = list(executor.map(simulated_io, range(3)))
threaded_elapsed = perf_counter() - started

assert sequential == concurrent
print(f"同步耗时：{sequential_elapsed:.2f} 秒")
print(f"线程池耗时：{threaded_elapsed:.2f} 秒")

同步耗时：0.60 秒
线程池耗时：0.21 秒


## 3. 共享状态与竞态条件

下面故意把“读取—等待—写回”拆开，让多个线程读取到相同旧值。GIL 不能替代业务层的锁：只要一个逻辑操作包含多个步骤，就可能在步骤之间发生线程切换。

In [19]:
from concurrent.futures import ThreadPoolExecutor
from time import sleep


unsafe_counter = 0


def unsafe_increment() -> None:
    global unsafe_counter
    snapshot = unsafe_counter
    sleep(0.005)  # 强制扩大竞态窗口，仅用于教学
    unsafe_counter = snapshot + 1


with ThreadPoolExecutor(max_workers=20) as executor:
    list(executor.map(lambda _: unsafe_increment(), range(20)))

print(f"无锁结果：{unsafe_counter}（期望 20，通常会丢失更新）")
assert unsafe_counter < 20

无锁结果：1（期望 20，通常会丢失更新）


In [20]:
safe_counter = 0
counter_lock = threading.Lock()


def safe_increment() -> None:
    global safe_counter
    with counter_lock:  # 相当于 Java 中保护同一临界区的锁
        snapshot = safe_counter
        sleep(0.001)
        safe_counter = snapshot + 1


with ThreadPoolExecutor(max_workers=20) as executor:
    list(executor.map(lambda _: safe_increment(), range(20)))

print(f"加锁结果：{safe_counter}")
assert safe_counter == 20

加锁结果：20


## 4. 线程同步工具全景

线程同步工具主要解决两类问题：

1. **保护共享状态**：避免多个线程同时修改同一份数据，或等待共享状态满足某个条件。
2. **在线程间传递任务或信号**：让线程安全地交接工作、发送通知或在某个阶段集合。

| 常见概念 | Python 工具 | 适用场景 |
|---|---|---|
| 互斥锁 | `threading.Lock` | 同一时刻只允许一个线程访问共享状态 |
| 可重入锁 | `threading.RLock` | 同一线程可能通过嵌套调用重复获取同一把锁 |
| 条件变量 | `threading.Condition` | 等待“队列非空”“状态已改变”等条件 |
| 信号量 | `threading.Semaphore` | 限制同时访问资源的线程数量 |
| 有界信号量 | `threading.BoundedSemaphore` | 同上，并检测释放次数超过初始值的错误 |
| 广播通知 | `threading.Event` | 停止通知、配置加载完成、服务就绪通知 |
| 循环栅栏 | `threading.Barrier` | 固定数量线程在某个阶段互相等待 |
| 阻塞队列 | `queue.Queue` | 生产者与消费者之间安全传递任务和数据 |
| 线程池 | `ThreadPoolExecutor` | 管理工作线程并提交异步任务 |
| Future | `concurrent.futures.Future` | 表示异步任务将来的结果、异常或取消状态 |
| 线程局部变量 | `threading.local` | 每个线程保存一份互不影响的数据 |
| 原子计数器 | 标准库没有完全对应类 | 一般使用 `Lock` 保护普通整数及复合操作 |

优先使用 `with lock:` 等上下文管理器，确保异常发生时也能释放同步资源。

## 5. Lock 与 RLock：保护共享状态

`Lock` 是最基本的互斥锁。前面的安全计数器已经展示了它的用法。锁的范围应只覆盖真正需要保护的临界区，不要在持锁期间执行慢速网络请求。

`RLock` 是**可重入锁**：持有它的线程可以再次获取同一把锁，但必须释放相同次数。它适合一个加锁方法调用另一个也需要同一把锁的方法。

Java 对照：Python `RLock` 接近 Java `ReentrantLock`；Python 普通 `Lock` 不支持同一线程重复获取。

In [21]:
import threading


reentrant_lock = threading.RLock()
audit_log: list[str] = []


def append_detail() -> None:
    with reentrant_lock:  # 同一线程第二次获取，RLock 允许
        audit_log.append("detail")


def append_record() -> None:
    with reentrant_lock:
        audit_log.append("record")
        append_detail()


append_record()
print(audit_log)
assert audit_log == ["record", "detail"]

['record', 'detail']


## 6. Condition：等待状态发生变化

`Condition` 把一把锁和“条件已经改变”的通知结合起来。典型流程是：

- 消费者在持锁状态下检查条件；条件不满足时调用 `wait()` 或 `wait_for()`；
- `wait()` 会原子地释放锁并睡眠，醒来后重新获得锁；
- 生产者修改状态后调用 `notify()` 唤醒一个等待者，或 `notify_all()` 唤醒全部等待者；
- 检查条件必须使用循环或 `wait_for(predicate)`，因为线程醒来不代表条件一定仍然成立。

Java 对照：接近 `Condition.await()/signal()`，也类似在监视器上使用 `wait()/notify()`。

In [22]:
from concurrent.futures import ThreadPoolExecutor
from time import sleep


condition = threading.Condition()
condition_items: list[str] = []


def condition_consumer() -> str:
    with condition:
        condition.wait_for(lambda: bool(condition_items))
        return condition_items.pop(0)


def condition_producer() -> None:
    sleep(0.05)
    with condition:
        condition_items.append("新任务")
        condition.notify()


with ThreadPoolExecutor(max_workers=2) as executor:
    consumed_future = executor.submit(condition_consumer)
    executor.submit(condition_producer).result()
    print("消费者获得：", consumed_future.result())

消费者获得： 新任务


## 7. Semaphore 与 BoundedSemaphore：限制并发数量

信号量内部维护许可证数量。线程进入受限区域前获取一个许可证，离开时释放。它适合限制数据库连接、文件句柄或第三方接口的并发访问数量。

- `Semaphore(n)`：允许最多 `n` 个线程同时进入；普通信号量允许释放后超过初始值。
- `BoundedSemaphore(n)`：释放次数导致计数超过初始值时抛出 `ValueError`，更容易发现 acquire/release 不配对。

它们控制的是**同时进入的数量**，不是任务执行顺序，也不能保护多个步骤组成的共享状态更新。

In [23]:
limit = threading.Semaphore(2)
limit_state_lock = threading.Lock()
active_workers = 0
peak_workers = 0


def limited_resource(task_id: int) -> int:
    global active_workers, peak_workers
    with limit:
        with limit_state_lock:
            active_workers += 1
            peak_workers = max(peak_workers, active_workers)
        sleep(0.03)
        with limit_state_lock:
            active_workers -= 1
    return task_id


with ThreadPoolExecutor(max_workers=5) as executor:
    list(executor.map(limited_resource, range(5)))

print("峰值并发：", peak_workers)
assert peak_workers == 2

bounded = threading.BoundedSemaphore(1)
bounded.acquire()
bounded.release()
try:
    bounded.release()  # 没有对应的 acquire
except ValueError:
    print("BoundedSemaphore 发现了过度释放")

峰值并发： 2
BoundedSemaphore 发现了过度释放


## 8. Event 与 Barrier：发送信号和阶段集合

`Event` 保存一个布尔标志：

- `wait()` 等待标志变为真；
- `set()` 将标志设为真并唤醒所有等待线程；
- `clear()` 恢复为假；`is_set()` 查询状态。

它适合“一对多”的就绪或停止通知，不携带任务数据。Java 中没有完全相同的基础类，可把它理解为可重置的单次 `CountDownLatch`。

`Barrier(parties)` 让固定数量线程在某个阶段互相等待。全部到达后一起继续，并可在下一轮复用；这接近 Java `CyclicBarrier`。超时或某个线程失败时应处理 `BrokenBarrierError`。

In [24]:
ready_event = threading.Event()


def wait_for_initialization() -> str:
    ready_event.wait()
    return "初始化完成，工作线程继续"


with ThreadPoolExecutor(max_workers=1) as executor:
    waiting_future = executor.submit(wait_for_initialization)
    sleep(0.05)
    ready_event.set()  # 广播唤醒等待线程
    print(waiting_future.result())

初始化完成，工作线程继续


In [25]:
stage_barrier = threading.Barrier(3)


def finish_stage(worker_id: int) -> tuple[int, int]:
    sleep(worker_id * 0.02)
    arrival_token = stage_barrier.wait(timeout=1)
    return worker_id, arrival_token


with ThreadPoolExecutor(max_workers=3) as executor:
    barrier_results = list(executor.map(finish_stage, range(3)))

print("Barrier 结果：", barrier_results)
assert sorted(token for _, token in barrier_results) == [0, 1, 2]

Barrier 结果： [(0, 0), (1, 1), (2, 2)]


## 9. Queue：在线程间安全传递任务

`queue.Queue` 是线程安全阻塞队列，通常比“共享 list + Lock + Condition”更安全、更简单：

- `put()`：放入任务；队列有容量且已满时可以阻塞；
- `get()`：取出任务；队列为空时阻塞；
- `task_done()`：声明一次 `get()` 取得的任务已经处理完；
- `join()`：等待所有已入队任务都调用 `task_done()`；
- 可使用特殊哨兵值通知消费者退出。

Java 对照：接近 `BlockingQueue`，例如 `ArrayBlockingQueue` 或 `LinkedBlockingQueue`。

In [26]:
from queue import Queue


work_queue: Queue[int | None] = Queue(maxsize=3)
queue_results: list[int] = []
queue_results_lock = threading.Lock()


def queue_consumer() -> None:
    while True:
        item = work_queue.get()
        try:
            if item is None:  # 哨兵：通知消费者退出
                return
            with queue_results_lock:
                queue_results.append(item * item)
        finally:
            work_queue.task_done()  # 每次 get 必须对应一次 task_done


with ThreadPoolExecutor(max_workers=2) as executor:
    consumer_futures = [executor.submit(queue_consumer) for _ in range(2)]
    for value in range(5):
        work_queue.put(value)
    for _ in consumer_futures:
        work_queue.put(None)
    work_queue.join()
    for future in consumer_futures:
        future.result()

print("Queue 处理结果：", sorted(queue_results))
assert sorted(queue_results) == [0, 1, 4, 9, 16]

Queue 处理结果： [0, 1, 4, 9, 16]


## 10. ThreadPoolExecutor 与 Future

`ThreadPoolExecutor` 管理一组可复用工作线程，`submit()` 返回 `Future`：

- `result()` 等待并返回结果；任务失败时重新抛出异常；
- `exception()` 返回任务异常；
- `done()`、`running()`、`cancelled()` 查询状态；
- `cancel()` 只能取消尚未开始运行的任务；
- `add_done_callback()` 注册完成回调。

Java 对照：`ThreadPoolExecutor` 接近 `ExecutorService`，Python `Future` 接近 Java `Future`。它不是 `asyncio.Future`，两者由不同调度系统管理。

In [27]:
from concurrent.futures import Future


def divide(left: int, right: int) -> float:
    return left / right


with ThreadPoolExecutor(max_workers=2) as executor:
    successful_future: Future[float] = executor.submit(divide, 8, 2)
    failed_future: Future[float] = executor.submit(divide, 1, 0)

    print("Future 结果：", successful_future.result())
    print("Future 异常：", type(failed_future.exception()).__name__)

assert successful_future.done()
assert isinstance(failed_future.exception(), ZeroDivisionError)

Future 结果： 4.0
Future 异常： ZeroDivisionError


## 11. threading.local 与线程安全计数器

`threading.local()` 创建线程局部存储。同一个对象上的属性在不同线程中拥有独立值，适合保存请求 ID、日志上下文或不能跨线程共享的客户端状态。它接近 Java `ThreadLocal`。

线程池会复用线程，因此线程局部数据也可能残留到下一个任务；任务结束时应清理不应复用的值，不能把它当作请求级全局变量。

Python 标准库没有与 Java `AtomicInteger` 完全对应的通用原子计数器。不要依赖某个表达式“碰巧在当前 CPython 中看起来原子”；需要稳定语义时，用 `Lock` 把读取、修改和写回作为一个整体保护起来。

In [28]:
thread_context = threading.local()


def handle_request(request_id: str) -> tuple[str, str]:
    thread_context.request_id = request_id
    sleep(0.02)
    result = (threading.current_thread().name, thread_context.request_id)
    del thread_context.request_id  # 线程池复用线程，主动清理
    return result


with ThreadPoolExecutor(max_workers=2, thread_name_prefix="worker") as executor:
    print("线程局部数据：", list(executor.map(handle_request, ("req-A", "req-B"))))


class AtomicCounter:
    def __init__(self) -> None:
        self._value = 0
        self._lock = threading.Lock()

    def increment(self) -> None:
        with self._lock:
            self._value += 1

    @property
    def value(self) -> int:
        with self._lock:
            return self._value


atomic_counter = AtomicCounter()
with ThreadPoolExecutor(max_workers=8) as executor:
    list(executor.map(lambda _: atomic_counter.increment(), range(100)))

print("计数器结果：", atomic_counter.value)
assert atomic_counter.value == 100

线程局部数据： [('worker_0', 'req-A'), ('worker_1', 'req-B')]
计数器结果： 100


### 同步工具选择顺序

1. 只是向工作线程提交独立任务：`ThreadPoolExecutor`。
2. 在线程间传递任务或数据：优先 `Queue`。
3. 保护一小段共享状态：`Lock`；存在嵌套加锁调用才使用 `RLock`。
4. 等待共享状态满足复杂条件：`Condition`；如果只是队列非空，优先直接用 `Queue`。
5. 限制同时使用某资源的线程数量：`Semaphore` 或更严格的 `BoundedSemaphore`。
6. 广播“可以开始/应该停止”：`Event`。
7. 固定数量线程需要阶段性集合：`Barrier`。

高层工具已经封装好锁和等待逻辑时，优先使用高层工具；不要重复手写一套脆弱的同步协议。

## 12. GIL 到底限制了什么

在项目当前使用的**标准 CPython 3.13 构建**中，全局解释器锁（GIL）通常只允许一个线程在同一时刻执行 Python 字节码。因此：

- I/O 密集型任务仍能从线程受益，因为等待期间线程会释放执行机会；
- 纯 Python CPU 密集型任务通常不能靠增加线程获得多核并行；
- NumPy 等原生扩展可能在计算时释放 GIL，此时线程仍可能并行；
- 锁仍然必要：GIL 不保证由多步组成的业务操作是原子的。

Python 3.13 也提供可选的 free-threaded 构建，但它不是本项目当前的默认解释器，不能据此忽略同步问题。

## 13. CPU 密集型任务使用进程池

每个进程拥有独立的解释器和内存空间，因此能够利用多个 CPU 核心。代价是启动更重、数据需要序列化、进程间不能随意共享对象。

完整示例位于 `examples/02_process_pool.py`。关键规则：提交给进程池的函数应定义在模块顶层，并使用 `if __name__ == "__main__"` 保护程序入口。

In [29]:
from pathlib import Path
import subprocess
import sys


script = Path("examples/02_process_pool.py")
completed = subprocess.run(
    [sys.executable, str(script)],
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

{20000: 2262, 22000: 2464}
预期：两个计算任务由不同进程执行，不共享同一个 GIL。



直接在 Notebook 单元格中定义进程任务，在某些平台和启动方式下可能无法被子进程导入。因此教程把完整多进程示例放进 `.py` 文件，这也更接近生产代码的组织方式。

## 14. 线程与进程对比

| 特征 | 线程 | 进程 |
|---|---|---|
| 内存 | 同一进程内共享 | 默认相互隔离 |
| 创建成本 | 较低 | 较高 |
| 数据传递 | 直接共享对象，但需要锁 | 通常需要序列化/IPC |
| 纯 Python CPU 并行 | 默认受 GIL 限制 | 可以利用多核 |
| 典型用途 | 阻塞式 I/O | CPU 密集型计算 |

不要把进程池用于大量极短任务：启动和序列化开销可能比计算本身还大。

## 15. 与 Java 的差异

- Java 多线程 CPU 计算通常可以直接利用多核；默认 CPython 线程执行纯 Python 字节码时要考虑 GIL。
- Java 对共享内存使用 `synchronized`、`Lock`、原子类等；Python 对应 `Lock`、`RLock`、Queue 等。
- Python 的进程池更常用于绕过默认 GIL 完成 CPU 并行，而 Java 通常不需要用多进程解决同一问题。
- Java 虚拟线程仍采用线程式阻塞编程模型；Python `asyncio` 采用显式 `async`/`await` 协作式调度。

## 16. 常见误区与练习

**误区：**有 GIL 就不需要 Lock；线程越多越快；进程可以零成本共享任意对象；Notebook 中能运行的进程代码必然能作为脚本运行，反之亦然。

**练习：**先运行 `examples/06_thread_synchronization.py`，再修改 `examples/01_sync_and_threads.py` 的任务数和 `max_workers`，观察耗时变化。思考为什么线程数超过任务数后不再有明显收益。

下一章进入 `asyncio`：它不需要为每个等待任务分配一个操作系统线程。